# Questão 3 — Pós-Treino com LoRA/QLoRA — Qwen2.5-0.5B
### Dataset: docentesDC · Pares Instruction/Input/Output

Repete o experimento da Questão 2 (SFT completo), agora usando **LoRA** (Low-Rank Adaptation) para o fine-tuning. Em vez de atualizar todos os pesos do modelo, o LoRA congela o modelo base e treina apenas pequenas matrizes adaptadoras inseridas nas camadas de atenção — drasticamente mais leve em VRAM e em tempo de treino.

Avaliação ANTES e DEPOIS com as mesmas métricas da Questão 2: Entropia Cruzada, Perplexidade e Acurácia de previsão de tokens.

## Célula 1 — Instalação

In [1]:
!pip install -q "transformers>=4.45.0" "datasets>=2.20.0" "accelerate>=0.34.0" "peft>=0.13.0" "bitsandbytes>=0.43.0" tqdm


In [10]:
!pip install -U bitsandbytes

## Célula 2 — Imports, Configurações e Carregamento do Modelo Base

In [ ]:
import re, math, json, random, copy
import torch
from collections import Counter
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# ── Reprodutibilidade ──────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# ── Configurações ──────────────────────────────────────────────────────────
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "Qwen/Qwen2.5-0.5B"   # mesmo modelo da Questão 2, para comparação direta
MAX_LEN    = 256
EPOCHS     = 3
LR         = 2e-4                  # LR mais alto que o Full SFT (5e-5) — comum em LoRA

# ── Liga/desliga QLoRA (carrega o modelo em 4-bit) ──────────────────────────
USAR_QLORA = True   # True = QLoRA (4-bit, economiza mais VRAM) | False = LoRA puro [(fp16/bf16)

print(f"Dispositivo : {DEVICE}")
print(f"Modelo      : {MODEL_NAME}")
print(f"Modo        : {'QLoRA (4-bit)' if USAR_QLORA else 'LoRA (fp16)'}")

# ── Carrega tokenizer ────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ── Carrega modelo base (quantizado em 4-bit se QLoRA estiver ativo) ───────
if USAR_QLORA and DEVICE.type == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, trust_remote_code=True,
        quantization_config=bnb_config,
        device_map={"": 0},
    )
else:
    if USAR_QLORA:
        print("⚠️  QLoRA exige GPU CUDA — caindo para LoRA padrão em CPU/fp32.")
    model_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model_base.to(DEVICE)

n_params = sum(p.numel() for p in model_base.parameters())
print(f"Parâmetros  : {n_params/1e6:.1f}M")
print(f"Vocabulário : {tokenizer.vocab_size:,} tokens")


c:\Users\oscar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo : cuda
Modelo      : Qwen/Qwen2.5-0.5B
Modo        : QLoRA (4-bit)


W0620 19:07:34.377000 51276 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\oscar\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Parâmetros  : 315.1M
Vocabulário : 151,643 tokens


## Célula 3 — Pares Instruction/Input/Output (docentesDC)

> Reaproveita o mesmo arquivo de pares gerado na Questão 2 (`dados_sft_docentesDC.json`), garantindo dataset idêntico para comparação justa entre Full SFT (Q2) e LoRA/QLoRA (Q3).

In [3]:
import json

CAMINHO_DADOS = "dados_sft_docentesDC.json"   # gerado na Questão 2

with open(CAMINHO_DADOS, encoding="utf-8") as f:
    dados_sft = json.load(f)

print(f"Total de pares carregados: {len(dados_sft):,}")
print("\nExemplo:")
print(json.dumps(dados_sft[0], ensure_ascii=False, indent=2))


Total de pares carregados: 3,111

Exemplo:
{
  "instruction": "Explique o que é tratado neste trecho do material do professor Weslley Emmanuel Martins Lima.",
  "input": "",
  "output": "PonteiroPonteiros são tipos especiais de dados que armazenam endereços de memória.Uma variável do tipo ponteiro deve ser declarada da seguinte forma:tipo *nome_variavel;A variável ponteiro armazenará um endereço de memória de uma outra variável do tipo especificado.Exemplo: int *mema;memaarmazena endereço de memória de variáveis do tipo int."
}


## Célula 4 — Formatação dos Pares no Padrão de Prompt (Alpaca-style)

In [4]:
def formatar_prompt(exemplo):
    """
    Formata cada par instruction/input/output no estilo Alpaca,
    padrão amplamente usado para SFT de LLMs causais.
    """
    if exemplo.get("input", "").strip():
        prompt = (
            f"### Instrução:\n{exemplo['instruction']}\n\n"
            f"### Entrada:\n{exemplo['input']}\n\n"
            f"### Resposta:\n{exemplo['output']}"
        )
    else:
        prompt = (
            f"### Instrução:\n{exemplo['instruction']}\n\n"
            f"### Resposta:\n{exemplo['output']}"
        )
    return prompt + tokenizer.eos_token

textos_formatados = [formatar_prompt(ex) for ex in dados_sft]

print("=== Exemplo formatado ===")
print(textos_formatados[0])
print(f"\nTotal formatado: {len(textos_formatados):,}")


=== Exemplo formatado ===
### Instrução:
Explique o que é tratado neste trecho do material do professor Weslley Emmanuel Martins Lima.

### Resposta:
PonteiroPonteiros são tipos especiais de dados que armazenam endereços de memória.Uma variável do tipo ponteiro deve ser declarada da seguinte forma:tipo *nome_variavel;A variável ponteiro armazenará um endereço de memória de uma outra variável do tipo especificado.Exemplo: int *mema;memaarmazena endereço de memória de variáveis do tipo int.<|endoftext|>

Total formatado: 3,111


## Célula 5 — Split Treino/Avaliação

In [5]:
random.seed(42)
random.shuffle(textos_formatados)

N_EVAL = max(50, int(len(textos_formatados) * 0.1))   # 10% para avaliação (mín. 50)

textos_eval   = textos_formatados[:N_EVAL]
textos_treino = textos_formatados[N_EVAL:]

print(f"Treino    : {len(textos_treino):,}")
print(f"Avaliação : {len(textos_eval):,}")


Treino    : 2,800
Avaliação : 311


## Célula 6 — Função de Avaliação Quantitativa (PPL, Entropia, Acurácia)

In [6]:
def calcular_metricas(modelo, textos, batch_size=8):
    """
    Calcula Entropia Cruzada, Perplexidade e Acurácia de previsão de tokens
    — as três métricas pedidas no enunciado.
    """
    modelo.eval()
    total_loss   = 0.0
    total_tokens = 0
    total_certos = 0

    for i in range(0, len(textos), batch_size):
        batch = textos[i : i + batch_size]

        enc = tokenizer(
            batch, return_tensors="pt",
            max_length=MAX_LEN, truncation=True, padding="max_length"
        )
        input_ids      = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100   # ignora padding no loss

        with torch.no_grad():
            out = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        n_tok = attention_mask.sum().item()
        total_loss   += out.loss.item() * n_tok
        total_tokens += n_tok

        # Acurácia: pred[t] vs token real[t+1]
        preds      = out.logits.argmax(dim=-1)
        mask_shift = attention_mask[:, 1:].bool()
        certos     = (preds[:, :-1] == input_ids[:, 1:])[mask_shift].sum().item()
        total_certos += certos

    ec  = total_loss / total_tokens
    ppl = math.exp(ec)
    acc = total_certos / total_tokens

    return {
        "entropia_cruzada": round(ec, 4),
        "perplexidade":     round(ppl, 2),
        "acuracia_tokens":  round(acc * 100, 2)
    }

print("Função de avaliação definida.")


Função de avaliação definida.


## Célula 7 — Perguntas de Teste (geração qualitativa antes/depois)

In [7]:
# Mesmas perguntas da Questão 2, para comparação direta entre Full SFT e LoRA.
perguntas_teste = [
    "Quem é o professor responsável pela disciplina de Tópicos em IA?",
    "Qual é a titulação do professor X do Departamento de Computação?",
    "Em que área de pesquisa atua o professor Y?",
]

def gerar_resposta(modelo, instrucao, max_new_tokens=80):
    modelo.eval()
    prompt = f"### Instrução:\n{instrucao}\n\n### Resposta:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = modelo.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    texto = tokenizer.decode(out[0], skip_special_tokens=True)
    return texto.split("### Resposta:")[-1].strip()

print("Funções de geração definidas.")


Funções de geração definidas.


## Célula 8 — Avaliação ANTES do LoRA/QLoRA

In [8]:
print("Calculando métricas do modelo base (sem fine-tuning)...")
metricas_antes = calcular_metricas(model_base, textos_eval)

print("\n=== ANTES do LoRA/QLoRA ===")
print(f"  Entropia Cruzada : {metricas_antes['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_antes['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_antes['acuracia_tokens']}%")

print("\n--- Respostas ANTES do treino ---")
for p in perguntas_teste:
    print(f"\nPergunta : {p}")
    print(f"Resposta : {gerar_resposta(model_base, p)}")


Calculando métricas do modelo base (sem fine-tuning)...

=== ANTES do LoRA/QLoRA ===
  Entropia Cruzada : 3.5413
  Perplexidade     : 34.51
  Acurácia Tokens  : 38.42%

--- Respostas ANTES do treino ---

Pergunta : Quem é o professor responsável pela disciplina de Tópicos em IA?


c:\Users\oscar\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Resposta : Os professores responsáveis pela disciplina de Tópicos em IA são os professores do departamento de Inteligência Artificial, por exemplo, Prof. [nome] ou [professora].

Pergunta : Qual é a titulação do professor X do Departamento de Computação?
Resposta : X é um Professor da Faculdade de Informática (UF/MA) na Universidade Federal do Maranhão, com uma Mestranda em Engenharia de Software pela Universidade Federal do Rio de Janeiro (UNIFIORES).

Pergunta : Em que área de pesquisa atua o professor Y?
Resposta : A pesquisa da área de pesquisa do professor Y é a pesquisa sobre os efeitos da estrutura física na preservação da saúde, principalmente em um contexto mundialmente globalizado como a pandemia da Covid-19. O professor Y tem interesse em entender as influências da estrutura física no bemestar da população mundial e também no desenvolvimento das áreas físicas.

Neste contexto,


## Célula 9 — Configuração do LoRA (PEFT)

Diferente da Questão 2, **não copiamos o modelo inteiro** com `deepcopy`. Em vez disso, o `model_base` é envolvido pelo PEFT, que insere as matrizes adaptadoras (A e B) nas camadas de projeção de atenção (`q_proj`, `k_proj`, `v_proj`, `o_proj`) e congela todo o resto. Isso reduz drasticamente o uso de VRAM em comparação ao Full SFT da Questão 2.

In [10]:
# Se QLoRA: prepara o modelo quantizado para treino (cast de norm layers, etc.)
if USAR_QLORA and DEVICE.type == "cuda":
    model_base = prepare_model_for_kbit_training(model_base)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # rank das matrizes adaptadoras (8–32 é o range usual)
    lora_alpha=32,            # fator de escala (geralmente 2x o rank)
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # camadas de atenção do Qwen2.5
)

model_ft = get_peft_model(model_base, lora_config)

# ── Mostra quantos parâmetros são realmente treináveis ──────────────────────
model_ft.print_trainable_parameters()


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


c:\Users\oscar\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\oscar\AppData\Local\Programs\Python\Python312\Lib\site-packages\peft\tuners\tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## Célula 10 — Dataset de Treino e Fine-Tuning com LoRA

In [ ]:
from transformers import TrainerCallback
import time

class SFTDataset(Dataset):
    def __init__(self, textos):
        self.enc = tokenizer(
            textos, return_tensors="pt",
            max_length=MAX_LEN, truncation=True, padding="max_length"
        )

    def __len__(self):
        return self.enc["input_ids"].shape[0]

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx]
        }

print("Tokenizando dados de treino...")
train_ds = SFTDataset(textos_treino)
print("Tokenizando dados de avaliação...")
eval_ds  = SFTDataset(textos_eval)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f"Treino: {len(train_ds):,} | Avaliação: {len(eval_ds):,}")

OUTPUT_DIR = "./qwen25_lora_docentes"

# ── Callback para mostrar progresso em tempo real, passo a passo ───────────
class ProgressoTempoReal(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.inicio = time.time()
        total_passos = state.max_steps
        print(f"Treino iniciado — {total_passos} passos totais\n")

    def on_step_end(self, args, state, control, **kwargs):
        passo  = state.global_step
        total  = state.max_steps
        decorrido = time.time() - self.inicio
        passos_por_seg = passo / decorrido if decorrido > 0 else 0
        restante = (total - passo) / passos_por_seg if passos_por_seg > 0 else 0

        pct = (passo / total) * 100
        print(
            f"\r[{passo:>4}/{total}] {pct:5.1f}%  "
            f"| decorrido: {decorrido/60:5.1f}min  "
            f"| restante: ~{restante/60:5.1f}min",
            end="", flush=True
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            print(f"\n  passo {state.global_step}: loss = {logs['loss']:.4f}")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"\n>>> Época {int(state.epoch)} concluída <<<\n")

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,     # LoRA usa bem menos VRAM -> batch maior que o Full SFT
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,     # batch efetivo = 16
    gradient_checkpointing=True,

    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    fp16=(DEVICE.type == "cuda" and not USAR_QLORA),  # QLoRA já opera em 4-bit/fp16 internamente
    dataloader_num_workers=0,          # evita travamento de multiprocessing no Windows

    logging_steps=5,
    logging_first_step=True,
    disable_tqdm=False,
    report_to="none"
)

trainer = Trainer(
    model=model_ft,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    callbacks=[ProgressoTempoReal()],
)

# ── Confirma se está rodando na GPU antes de começar ────────────────────────
print("CUDA disponível:", torch.cuda.is_available())
print("Device do modelo:", next(model_ft.parameters()).device)
print()

print(f"Iniciando LoRA/QLoRA SFT — {EPOCHS} épocas em {DEVICE}...\n")
resultado = trainer.train()

print("\n\nTreinamento concluído!")
print(f"  Loss final de treino : {resultado.training_loss:.4f}")
print(f"  Tempo total          : {resultado.metrics['train_runtime']:.0f}s")


Tokenizando dados de treino...
Tokenizando dados de avaliação...
Treino: 2,800 | Avaliação: 311
CUDA disponível: True
Device do modelo: cuda:0

Iniciando LoRA/QLoRA SFT — 3 épocas em cuda...

Treino iniciado — 525 passos totais



  0%|          | 0/525 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


[   1/525]   0.2%  | decorrido:   0.0min  | restante: ~ 21.4min

  0%|          | 1/525 [00:02<21:19,  2.44s/it]


  passo 1: loss = 3.2803
{'loss': 3.2803, 'grad_norm': 1.470426082611084, 'learning_rate': 7.4074074074074075e-06, 'epoch': 0.01}
[   2/525]   0.4%  | decorrido:   0.1min  | restante: ~ 19.3min

...

 93%|█████████▎| 486/525 [16:11<01:32,  2.36s/it]

[ 487/525]  92.8%  | decorrido:  16.2min  | restante: ~  1.3min

 93%|█████████▎| 487/525 [16:13<01:27,  2.31s/it]

[ 488/525]  93.0%  | decorrido:  16.3min  | restante: ~  1.2min

 93%|█████████▎| 488/525 [16:16<01:29,  2.42s/it]

## Célula 11 — Avaliação DEPOIS do LoRA/QLoRA

In [ ]:
print("Calculando métricas do modelo após LoRA/QLoRA...")
metricas_depois = calcular_metricas(model_ft, textos_eval)

print("\n=== DEPOIS do LoRA/QLoRA ===")
print(f"  Entropia Cruzada : {metricas_depois['entropia_cruzada']}")
print(f"  Perplexidade     : {metricas_depois['perplexidade']}")
print(f"  Acurácia Tokens  : {metricas_depois['acuracia_tokens']}%")

print("\n--- Respostas DEPOIS do treino ---")
for p in perguntas_teste:
    print(f"\nPergunta : {p}")
    print(f"Resposta : {gerar_resposta(model_ft, p)}")


## Célula 12 — Comparação Final ANTES vs. DEPOIS

In [ ]:
print("=" * 62)
print("    COMPARAÇÃO: ANTES vs. DEPOIS DO LoRA/QLoRA")
print("=" * 62)
print(f"{'Métrica':<28} {'Antes':>10} {'Depois':>10} {'Δ':>10}")
print("-" * 62)

d_ec  = metricas_depois["entropia_cruzada"] - metricas_antes["entropia_cruzada"]
d_ppl = metricas_depois["perplexidade"]     - metricas_antes["perplexidade"]
d_acc = metricas_depois["acuracia_tokens"]  - metricas_antes["acuracia_tokens"]

print(f"{'Entropia Cruzada':<28} {metricas_antes['entropia_cruzada']:>10.4f} {metricas_depois['entropia_cruzada']:>10.4f} {d_ec:>+10.4f}")
print(f"{'Perplexidade (PPL)':<28} {metricas_antes['perplexidade']:>10.2f} {metricas_depois['perplexidade']:>10.2f} {d_ppl:>+10.2f}")
print(f"{'Acurácia de Tokens (%)':<28} {metricas_antes['acuracia_tokens']:>10.2f} {metricas_depois['acuracia_tokens']:>10.2f} {d_acc:>+10.2f}")
print("=" * 62)

var_ppl = (d_ppl / metricas_antes["perplexidade"]) * 100
print(f"\nVariação relativa da Perplexidade: {var_ppl:+.1f}%")
if d_ppl < 0:
    print("✓ Perplexidade REDUZIU — modelo melhorou no domínio docentesDC.")
else:
    print("✗ Perplexidade aumentou — revisar hiperparâmetros ou dados de treino.")

print("\n--- Comparação lado a lado das respostas geradas ---")
for p in perguntas_teste:
    print(f"\nPergunta : {p}")
    print(f"  ANTES  : {gerar_resposta(model_base, p)}")
    print(f"  DEPOIS : {gerar_resposta(model_ft, p)}")


## Célula 13 — (Opcional) Comparação com o Full SFT da Questão 2

Se você tiver o arquivo `resultados_sft_q2.json` salvo na Questão 2, esta célula carrega os resultados do Full SFT e compara lado a lado com o LoRA/QLoRA — útil para a análise pedida no relatório (qual técnica trouxe melhor custo-benefício).

In [ ]:
import os

CAMINHO_Q2 = "resultados_sft_q2.json"

if os.path.exists(CAMINHO_Q2):
    with open(CAMINHO_Q2, encoding="utf-8") as f:
        resultados_q2 = json.load(f)

    print("=" * 70)
    print("   COMPARAÇÃO: Full SFT (Q2)  vs.  LoRA/QLoRA (Q3)")
    print("=" * 70)
    print(f"{'Métrica (depois)':<28} {'Full SFT':>15} {'LoRA/QLoRA':>15}")
    print("-" * 70)
    print(f"{'Perplexidade':<28} {resultados_q2['metricas_depois']['perplexidade']:>15.2f} {metricas_depois['perplexidade']:>15.2f}")
    print(f"{'Entropia Cruzada':<28} {resultados_q2['metricas_depois']['entropia_cruzada']:>15.4f} {metricas_depois['entropia_cruzada']:>15.4f}")
    print(f"{'Acurácia Tokens (%)':<28} {resultados_q2['metricas_depois']['acuracia_tokens']:>15.2f} {metricas_depois['acuracia_tokens']:>15.2f}")
else:
    print(f"Arquivo {CAMINHO_Q2} não encontrado — pule esta célula ou rode a Questão 2 primeiro.")


## Célula 14 — Salvar Adaptadores LoRA e Resultados

In [ ]:
# Salva SOMENTE os adaptadores LoRA (poucos MB), não o modelo inteiro.
# Para recarregar depois: PeftModel.from_pretrained(model_base, "./qwen25_lora_docentes_final")
model_ft.save_pretrained("./qwen25_lora_docentes_final")
tokenizer.save_pretrained("./qwen25_lora_docentes_final")
print("Adaptadores LoRA salvos em ./qwen25_lora_docentes_final")

resultados = {
    "modelo": MODEL_NAME,
    "tecnica": "QLoRA (4-bit)" if USAR_QLORA else "LoRA",
    "dataset": "docentesDC",
    "n_treino": len(textos_treino),
    "n_avaliacao": len(textos_eval),
    "hiperparametros": {
        "epocas": args.num_train_epochs,
        "batch_efetivo": args.per_device_train_batch_size * args.gradient_accumulation_steps,
        "learning_rate": args.learning_rate,
        "max_length": MAX_LEN,
        "lora_r": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "lora_target_modules": lora_config.target_modules,
    },
    "metricas_antes": metricas_antes,
    "metricas_depois": metricas_depois,
    "delta": {
        "entropia_cruzada": round(d_ec, 4),
        "perplexidade": round(d_ppl, 2),
        "acuracia_tokens_pp": round(d_acc, 2)
    }
}

with open("resultados_lora_q3.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2, default=str)
print("Métricas salvas em resultados_lora_q3.json")

print("\n=== RESUMO FINAL ===")
print(f"Modelo base -> PPL: {metricas_antes['perplexidade']} | EC: {metricas_antes['entropia_cruzada']} | Acc: {metricas_antes['acuracia_tokens']}%")
print(f"Modelo LoRA -> PPL: {metricas_depois['perplexidade']} | EC: {metricas_depois['entropia_cruzada']} | Acc: {metricas_depois['acuracia_tokens']}%")
